In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# ============================================================================
# CONFIGURACIÓN DEL ADC
# ============================================================================
N_BITS = 12
VREF = 1.8  # Voltaje de referencia diferencial total
VCM = 0.9   # Modo común
LSB = VREF / (2**N_BITS)  # Tamaño de 1 LSB = 440 uV

print("="*70)
print("ANÁLISIS DNL/INL - ADC SAR 12-bit")
print("="*70)
print(f"Resolución: {N_BITS} bits")
print(f"Rango diferencial: ±{VREF/2} V")
print(f"LSB ideal: {LSB*1e6:.2f} µV")
print(f"Códigos totales: {2**N_BITS} (-2048 a +2047)")
print("="*70)

# ============================================================================
# LEER DATOS DE NGSPICE
# ============================================================================
print("\nLeyendo archivo dnl_inl_data.raw...")

# Aquí debes usar ltspice o leer el raw directamente
# Por simplicidad, asumo que tienes los datos en numpy arrays
# Si usas ltspice:
try:
    import ltspice
    l = ltspice.Ltspice('dnl_inl_data.raw')
    l.parse()
    
    vdiff = l.get_data('vdiff_vec')
    b0 = l.get_data('b0_vec')
    b1 = l.get_data('b1_vec')
    b2 = l.get_data('b2_vec')
    b3 = l.get_data('b3_vec')
    b4 = l.get_data('b4_vec')
    b5 = l.get_data('b5_vec')
    b6 = l.get_data('b6_vec')
    b7 = l.get_data('b7_vec')
    b8 = l.get_data('b8_vec')
    b9 = l.get_data('b9_vec')
    b10 = l.get_data('b10_vec')
    b11 = l.get_data('b11_vec')
    eoc = l.get_data('eoc_vec')
    
    print(f"✓ Datos cargados: {len(vdiff)} puntos")
    
except Exception as e:
    print(f"Error al leer con ltspice: {e}")
    print("Generando datos de ejemplo para demostración...")
    # Datos de ejemplo (reemplazar con tus datos reales)
    vdiff = np.linspace(-VREF, VREF, 4096)
    # Simular códigos digitales con algo de no-linealidad
    codes_ideal = np.linspace(-2048, 2047, 4096)
    codes = codes_ideal + np.random.randn(4096) * 0.5  # Agregar ruido
    codes = np.clip(codes, -2048, 2047).astype(int)

# ============================================================================
# CONVERTIR BITS A CÓDIGO DIGITAL BIPOLAR
# ============================================================================
print("\nConvirtiendo bits a código digital bipolar...")

# Fórmula bipolar: -b0*2048 + b1*1024 + b2*512 + ... + b11*1
# b0 es el bit de signo (MSB), b11 es el LSB
def bits_to_code(b0, b1, b2, b3, b4, b5, b6, b7, b8, b9, b10, b11):
    """
    Convierte los 12 bits a código bipolar (-2048 a +2047)
    b0 = MSB (signo), b11 = LSB
    """
    # Convertir a binario (asumiendo que los bits son voltajes: >0.9V = 1, <0.9V = 0)
    bits = np.array([b0, b1, b2, b3, b4, b5, b6, b7, b8, b9, b10, b11])
    bits = (bits > 0.9).astype(int)
    
    # Pesos para código bipolar
    weights = np.array([-2048, 1024, 512, 256, 128, 64, 32, 16, 8, 4, 2, 1])
    
    code = np.sum(bits.T * weights, axis=1)
    return code.astype(int)

try:
    codes = bits_to_code(b0, b1, b2, b3, b4, b5, b6, b7, b8, b9, b10, b11)
    print(f"✓ Códigos convertidos: min={codes.min()}, max={codes.max()}")
except:
    print("Usando códigos de ejemplo...")

# ============================================================================
# ANÁLISIS DE CÓDIGOS
# ============================================================================
print("\n" + "="*70)
print("ANÁLISIS DE CÓDIGOS")
print("="*70)

# Contar ocurrencias de cada código
all_codes = np.arange(-2048, 2048)  # Todos los códigos posibles
code_counts = np.zeros(len(all_codes))

for i, code in enumerate(all_codes):
    code_counts[i] = np.sum(codes == code)

# Detectar códigos faltantes (missing codes)
missing_codes = all_codes[code_counts == 0]
print(f"Códigos faltantes (missing codes): {len(missing_codes)}")
if len(missing_codes) > 0 and len(missing_codes) <= 20:
    print(f"  Códigos: {missing_codes}")

# Códigos con más ocurrencias (wide codes)
mean_count = np.mean(code_counts[code_counts > 0])
wide_codes = all_codes[code_counts > mean_count * 2]
print(f"Códigos anchos (>2x promedio): {len(wide_codes)}")

# ============================================================================
# CÁLCULO DE DNL (Differential Non-Linearity)
# ============================================================================
print("\n" + "="*70)
print("CÁLCULO DE DNL")
print("="*70)

# DNL[k] = (ancho_real[k] - ancho_ideal) / LSB
# ancho_ideal = 1 LSB
# ancho_real[k] = número de apariciones del código k

ideal_count = len(vdiff) / len(all_codes)  # Ocurrencias ideales por código
DNL = (code_counts - ideal_count) / ideal_count  # En LSB

# Estadísticas DNL
DNL_max = np.max(DNL)
DNL_min = np.min(DNL)
DNL_rms = np.sqrt(np.mean(DNL**2))

print(f"DNL máximo: {DNL_max:+.3f} LSB")
print(f"DNL mínimo: {DNL_min:+.3f} LSB")
print(f"DNL RMS: {DNL_rms:.3f} LSB")

if DNL_max > 1.0:
    print(f"⚠️  ADVERTENCIA: DNL > +1 LSB detectado (códigos anchos)")
if DNL_min < -1.0:
    print(f"⚠️  ADVERTENCIA: DNL < -1 LSB detectado (missing codes)")

# ============================================================================
# CÁLCULO DE INL (Integral Non-Linearity)
# ============================================================================
print("\n" + "="*70)
print("CÁLCULO DE INL")
print("="*70)

# INL es la suma acumulada de DNL
INL = np.cumsum(DNL)

# Estadísticas INL
INL_max = np.max(INL)
INL_min = np.min(INL)
INL_rms = np.sqrt(np.mean(INL**2))

print(f"INL máximo: {INL_max:+.3f} LSB")
print(f"INL mínimo: {INL_min:+.3f} LSB")
print(f"INL RMS: {INL_rms:.3f} LSB")

# ============================================================================
# GRÁFICOS
# ============================================================================
print("\n" + "="*70)
print("GENERANDO GRÁFICOS")
print("="*70)

# Figura 1: Curva de transferencia
fig1, ax1 = plt.subplots(figsize=(14, 6))
ax1.plot(vdiff, codes, 'b.', markersize=2, alpha=0.5, label='Datos simulados')
ax1.plot(vdiff, vdiff / LSB, 'r--', linewidth=2, label='Ideal', alpha=0.7)
ax1.set_xlabel('Voltaje Diferencial (V)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Código Digital', fontsize=12, fontweight='bold')
ax1.set_title('Curva de Transferencia del ADC', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()
plt.tight_layout()
plt.show()

# Figura 2: Histograma de códigos
fig2, ax2 = plt.subplots(figsize=(14, 6))
ax2.bar(all_codes, code_counts, width=1, color='steelblue', alpha=0.7)
ax2.axhline(y=ideal_count, color='r', linestyle='--', linewidth=2, label=f'Ideal ({ideal_count:.1f})')
ax2.set_xlabel('Código Digital', fontsize=12, fontweight='bold')
ax2.set_ylabel('Ocurrencias', fontsize=12, fontweight='bold')
ax2.set_title('Histograma de Códigos', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')
ax2.legend()
plt.tight_layout()
plt.show()

print("✓ Histograma de códigos")
print("  Muestra la distribución de códigos digitales")
print("  - Barras iguales = ADC lineal ideal")
print("  - Barras faltantes = missing codes (DNL < -1)")
print("  - Barras muy altas = códigos anchos (DNL > +1)")

# Figura 3: DNL vs Código
fig3, ax3 = plt.subplots(figsize=(14, 6))
ax3.plot(all_codes, DNL, 'b-', linewidth=1.5)
ax3.axhline(y=0, color='g', linestyle='--', linewidth=2, label='Ideal (0 LSB)')
ax3.axhline(y=1, color='r', linestyle='--', linewidth=1, alpha=0.7, label='±1 LSB')
ax3.axhline(y=-1, color='r', linestyle='--', linewidth=1, alpha=0.7)
ax3.fill_between(all_codes, -1, 1, alpha=0.1, color='green', label='Zona aceptable')
ax3.set_xlabel('Código Digital', fontsize=12, fontweight='bold')
ax3.set_ylabel('DNL (LSB)', fontsize=12, fontweight='bold')
ax3.set_title('Differential Non-Linearity (DNL)', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.legend()
ax3.set_ylim([min(-2, DNL_min-0.5), max(2, DNL_max+0.5)])
plt.tight_layout()
plt.show()

print("\n✓ DNL (Differential Non-Linearity)")
print("  Mide la desviación del ancho de cada código respecto al ideal (1 LSB)")
print("  - DNL = 0: código perfecto")
print("  - DNL > +1: código muy ancho")
print("  - DNL < -1: missing code (código faltante)")

# Figura 4: INL vs Código
fig4, ax4 = plt.subplots(figsize=(14, 6))
ax4.plot(all_codes, INL, 'b-', linewidth=1.5)
ax4.axhline(y=0, color='g', linestyle='--', linewidth=2, label='Ideal (0 LSB)')
ax4.set_xlabel('Código Digital', fontsize=12, fontweight='bold')
ax4.set_ylabel('INL (LSB)', fontsize=12, fontweight='bold')
ax4.set_title('Integral Non-Linearity (INL)', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3)
ax4.legend()
plt.tight_layout()
plt.show()

print("\n✓ INL (Integral Non-Linearity)")
print("  Error acumulado respecto a la transferencia ideal")
print("  - INL = 0: perfectamente lineal")
print("  - INL positivo: ganancia mayor a la ideal")
print("  - INL negativo: ganancia menor a la ideal")
print("  - Forma de S: no-linealidad de segundo orden")
print("  - Forma de arco: bow error")

# ============================================================================
# RESUMEN FINAL
# ============================================================================
print("\n" + "="*70)
print("RESUMEN FINAL")
print("="*70)
print(f"LSB teórico: {LSB*1e6:.2f} µV")
print(f"Códigos simulados: {len(codes)}")
print(f"Códigos únicos: {len(np.unique(codes))}")
print(f"Códigos faltantes: {len(missing_codes)}")
print(f"\nDNL: min={DNL_min:+.3f} LSB, max={DNL_max:+.3f} LSB, RMS={DNL_rms:.3f} LSB")
print(f"INL: min={INL_min:+.3f} LSB, max={INL_max:+.3f} LSB, RMS={INL_rms:.3f} LSB")

# Evaluación de calidad
print(f"\n{'='*70}")
print("EVALUACIÓN DE CALIDAD:")
if abs(DNL_max) < 0.5 and abs(DNL_min) < 0.5:
    print("✓ DNL excelente (< ±0.5 LSB)")
elif abs(DNL_max) < 1.0 and abs(DNL_min) < 1.0:
    print("✓ DNL bueno (< ±1 LSB)")
else:
    print("✗ DNL requiere mejora (≥ ±1 LSB)")

if abs(INL_max) < 0.5 and abs(INL_min) < 0.5:
    print("✓ INL excelente (< ±0.5 LSB)")
elif abs(INL_max) < 1.0 and abs(INL_min) < 1.0:
    print("✓ INL bueno (< ±1 LSB)")
else:
    print("✗ INL requiere mejora (≥ ±1 LSB)")

print("="*70)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# ============================================================================
# CONFIGURACIÓN DEL ADC
# ============================================================================
N_BITS = 12
VREF = 1.8  # Voltaje de referencia diferencial total
VCM = 0.9   # Modo común
LSB = VREF / (2**N_BITS)  # Tamaño de 1 LSB = 440 uV

print("="*70)
print("ANÁLISIS DNL/INL - ADC SAR 12-bit")
print("="*70)
print(f"Resolución: {N_BITS} bits")
print(f"Rango diferencial: ±{VREF/2} V")
print(f"LSB ideal: {LSB*1e6:.2f} µV")
print(f"Códigos totales: {2**N_BITS} (-2048 a +2047)")
print("="*70)

# ============================================================================
# LEER DATOS DE NGSPICE
# ============================================================================
print("\nLeyendo archivo dnl_inl_data.raw...")

# Aquí debes usar ltspice o leer el raw directamente
# Por simplicidad, asumo que tienes los datos en numpy arrays
# Si usas ltspice:
try:
    import ltspice
    l = ltspice.Ltspice('dnl_inl_data.raw')
    l.parse()
    
    vdiff = l.get_data('vdiff_vec')
    b0 = l.get_data('b0_vec')
    b1 = l.get_data('b1_vec')
    b2 = l.get_data('b2_vec')
    b3 = l.get_data('b3_vec')
    b4 = l.get_data('b4_vec')
    b5 = l.get_data('b5_vec')
    b6 = l.get_data('b6_vec')
    b7 = l.get_data('b7_vec')
    b8 = l.get_data('b8_vec')
    b9 = l.get_data('b9_vec')
    b10 = l.get_data('b10_vec')
    b11 = l.get_data('b11_vec')
    eoc = l.get_data('eoc_vec')
    
    print(f"✓ Datos cargados: {len(vdiff)} puntos")
    
except Exception as e:
    print(f"Error al leer con ltspice: {e}")
    print("Generando datos de ejemplo para demostración...")
    # Datos de ejemplo (reemplazar con tus datos reales)
    vdiff = np.linspace(-VREF, VREF, 4096)
    # Simular códigos digitales con algo de no-linealidad
    codes_ideal = np.linspace(-2048, 2047, 4096)
    codes = codes_ideal + np.random.randn(4096) * 0.5  # Agregar ruido
    codes = np.clip(codes, -2048, 2047).astype(int)

# ============================================================================
# CONVERTIR BITS A CÓDIGO DIGITAL BIPOLAR
# ============================================================================
print("\nConvirtiendo bits a código digital bipolar...")

# Fórmula bipolar: -b0*2048 + b1*1024 + b2*512 + ... + b11*1
# b0 es el bit de signo (MSB), b11 es el LSB
def bits_to_code(b0, b1, b2, b3, b4, b5, b6, b7, b8, b9, b10, b11):
    """
    Convierte los 12 bits a código bipolar (-2048 a +2047)
    b0 = MSB (signo), b11 = LSB
    """
    # Convertir a binario (asumiendo que los bits son voltajes: >0.9V = 1, <0.9V = 0)
    bits = np.array([b0, b1, b2, b3, b4, b5, b6, b7, b8, b9, b10, b11])
    bits = (bits > 0.9).astype(int)
    
    # Pesos para código bipolar
    weights = np.array([-2048, 1024, 512, 256, 128, 64, 32, 16, 8, 4, 2, 1])
    
    code = np.sum(bits.T * weights, axis=1)
    return code.astype(int)

try:
    codes = bits_to_code(b0, b1, b2, b3, b4, b5, b6, b7, b8, b9, b10, b11)
    print(f"✓ Códigos convertidos: min={codes.min()}, max={codes.max()}")
except:
    print("Usando códigos de ejemplo...")

# ============================================================================
# ANÁLISIS DE CÓDIGOS
# ============================================================================
print("\n" + "="*70)
print("ANÁLISIS DE CÓDIGOS")
print("="*70)

# Contar ocurrencias de cada código
all_codes = np.arange(-2048, 2048)  # Todos los códigos posibles
code_counts = np.zeros(len(all_codes))

for i, code in enumerate(all_codes):
    code_counts[i] = np.sum(codes == code)

# Detectar códigos faltantes (missing codes)
missing_codes = all_codes[code_counts == 0]
print(f"Códigos faltantes (missing codes): {len(missing_codes)}")
if len(missing_codes) > 0 and len(missing_codes) <= 20:
    print(f"  Códigos: {missing_codes}")

# Códigos con más ocurrencias (wide codes)
mean_count = np.mean(code_counts[code_counts > 0])
wide_codes = all_codes[code_counts > mean_count * 2]
print(f"Códigos anchos (>2x promedio): {len(wide_codes)}")

# ============================================================================
# CÁLCULO DE DNL (Differential Non-Linearity)
# ============================================================================
print("\n" + "="*70)
print("CÁLCULO DE DNL")
print("="*70)

# DNL[k] = (ancho_real[k] - ancho_ideal) / LSB
# ancho_ideal = 1 LSB
# ancho_real[k] = número de apariciones del código k

ideal_count = len(vdiff) / len(all_codes)  # Ocurrencias ideales por código
DNL = (code_counts - ideal_count) / ideal_count  # En LSB

# Estadísticas DNL
DNL_max = np.max(DNL)
DNL_min = np.min(DNL)
DNL_rms = np.sqrt(np.mean(DNL**2))

print(f"DNL máximo: {DNL_max:+.3f} LSB")
print(f"DNL mínimo: {DNL_min:+.3f} LSB")
print(f"DNL RMS: {DNL_rms:.3f} LSB")

if DNL_max > 1.0:
    print(f"⚠️  ADVERTENCIA: DNL > +1 LSB detectado (códigos anchos)")
if DNL_min < -1.0:
    print(f"⚠️  ADVERTENCIA: DNL < -1 LSB detectado (missing codes)")

# ============================================================================
# CÁLCULO DE INL (Integral Non-Linearity)
# ============================================================================
print("\n" + "="*70)
print("CÁLCULO DE INL")
print("="*70)

# INL es la suma acumulada de DNL
INL = np.cumsum(DNL)

# Estadísticas INL
INL_max = np.max(INL)
INL_min = np.min(INL)
INL_rms = np.sqrt(np.mean(INL**2))

print(f"INL máximo: {INL_max:+.3f} LSB")
print(f"INL mínimo: {INL_min:+.3f} LSB")
print(f"INL RMS: {INL_rms:.3f} LSB")

# ============================================================================
# GRÁFICOS
# ============================================================================
print("\n" + "="*70)
print("GENERANDO GRÁFICOS")
print("="*70)

# Figura 1: Curva de transferencia
fig1, ax1 = plt.subplots(figsize=(14, 6))
ax1.plot(vdiff, codes, 'b.', markersize=2, alpha=0.5, label='Datos simulados')
ax1.plot(vdiff, vdiff / LSB, 'r--', linewidth=2, label='Ideal', alpha=0.7)
ax1.set_xlabel('Voltaje Diferencial (V)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Código Digital', fontsize=12, fontweight='bold')
ax1.set_title('Curva de Transferencia del ADC', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()
plt.tight_layout()
plt.show()

# Figura 2: Histograma de códigos
fig2, ax2 = plt.subplots(figsize=(14, 6))
ax2.bar(all_codes, code_counts, width=1, color='steelblue', alpha=0.7)
ax2.axhline(y=ideal_count, color='r', linestyle='--', linewidth=2, label=f'Ideal ({ideal_count:.1f})')
ax2.set_xlabel('Código Digital', fontsize=12, fontweight='bold')
ax2.set_ylabel('Ocurrencias', fontsize=12, fontweight='bold')
ax2.set_title('Histograma de Códigos', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')
ax2.legend()
plt.tight_layout()
plt.show()

print("✓ Histograma de códigos")
print("  Muestra la distribución de códigos digitales")
print("  - Barras iguales = ADC lineal ideal")
print("  - Barras faltantes = missing codes (DNL < -1)")
print("  - Barras muy altas = códigos anchos (DNL > +1)")

# Figura 3: DNL vs Código
fig3, ax3 = plt.subplots(figsize=(14, 6))
ax3.plot(all_codes, DNL, 'b-', linewidth=1.5)
ax3.axhline(y=0, color='g', linestyle='--', linewidth=2, label='Ideal (0 LSB)')
ax3.axhline(y=1, color='r', linestyle='--', linewidth=1, alpha=0.7, label='±1 LSB')
ax3.axhline(y=-1, color='r', linestyle='--', linewidth=1, alpha=0.7)
ax3.fill_between(all_codes, -1, 1, alpha=0.1, color='green', label='Zona aceptable')
ax3.set_xlabel('Código Digital', fontsize=12, fontweight='bold')
ax3.set_ylabel('DNL (LSB)', fontsize=12, fontweight='bold')
ax3.set_title('Differential Non-Linearity (DNL)', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.legend()
ax3.set_ylim([min(-2, DNL_min-0.5), max(2, DNL_max+0.5)])
plt.tight_layout()
plt.show()

print("\n✓ DNL (Differential Non-Linearity)")
print("  Mide la desviación del ancho de cada código respecto al ideal (1 LSB)")
print("  - DNL = 0: código perfecto")
print("  - DNL > +1: código muy ancho")
print("  - DNL < -1: missing code (código faltante)")

# Figura 4: INL vs Código
fig4, ax4 = plt.subplots(figsize=(14, 6))
ax4.plot(all_codes, INL, 'b-', linewidth=1.5)
ax4.axhline(y=0, color='g', linestyle='--', linewidth=2, label='Ideal (0 LSB)')
ax4.set_xlabel('Código Digital', fontsize=12, fontweight='bold')
ax4.set_ylabel('INL (LSB)', fontsize=12, fontweight='bold')
ax4.set_title('Integral Non-Linearity (INL)', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3)
ax4.legend()
plt.tight_layout()
plt.show()

print("\n✓ INL (Integral Non-Linearity)")
print("  Error acumulado respecto a la transferencia ideal")
print("  - INL = 0: perfectamente lineal")
print("  - INL positivo: ganancia mayor a la ideal")
print("  - INL negativo: ganancia menor a la ideal")
print("  - Forma de S: no-linealidad de segundo orden")
print("  - Forma de arco: bow error")

# ============================================================================
# RESUMEN FINAL
# ============================================================================
print("\n" + "="*70)
print("RESUMEN FINAL")
print("="*70)
print(f"LSB teórico: {LSB*1e6:.2f} µV")
print(f"Códigos simulados: {len(codes)}")
print(f"Códigos únicos: {len(np.unique(codes))}")
print(f"Códigos faltantes: {len(missing_codes)}")
print(f"\nDNL: min={DNL_min:+.3f} LSB, max={DNL_max:+.3f} LSB, RMS={DNL_rms:.3f} LSB")
print(f"INL: min={INL_min:+.3f} LSB, max={INL_max:+.3f} LSB, RMS={INL_rms:.3f} LSB")

# Evaluación de calidad
print(f"\n{'='*70}")
print("EVALUACIÓN DE CALIDAD:")
if abs(DNL_max) < 0.5 and abs(DNL_min) < 0.5:
    print("✓ DNL excelente (< ±0.5 LSB)")
elif abs(DNL_max) < 1.0 and abs(DNL_min) < 1.0:
    print("✓ DNL bueno (< ±1 LSB)")
else:
    print("✗ DNL requiere mejora (≥ ±1 LSB)")

if abs(INL_max) < 0.5 and abs(INL_min) < 0.5:
    print("✓ INL excelente (< ±0.5 LSB)")
elif abs(INL_max) < 1.0 and abs(INL_min) < 1.0:
    print("✓ INL bueno (< ±1 LSB)")
else:
    print("✗ INL requiere mejora (≥ ±1 LSB)")

print("="*70)